## Model Training — Booking Probability Prediction
Binary classifier estimating $P(Y=1|X)$ for integration into the ranking function:
$$FinalScore = (W_A \times BaseScore) + (W_B \times P(Y=1|X))$$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, classification_report, roc_curve, f1_score
)
import joblib

df = pd.read_csv("../data/processed/training_data.csv")
print(f"Dataset: {df.shape}")
df.head()

## Data Preprocessing

In [ ]:
# Drop columns not used for modeling
drop_cols = ['user_id', 'timestamp', 'listing_id']
df = df.drop(columns=drop_cols)

# Separate target
y = df['Y']
X = df.drop(columns=['Y'])

# Categoricals are already one-hot encoded from feature engineering
# Impute remaining missing values (numeric) with median
X = X.fillna(X.median())

print(f"Features: {X.shape[1]}")
print(f"Missing values remaining: {X.isnull().sum().sum()}")
X.head()

In [ ]:
# Train / test split (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train Y=1 ratio: {y_train.mean():.4f}")
print(f"Test  Y=1 ratio: {y_test.mean():.4f}")

## Model Training

In [ ]:
models = {
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, class_weight='balanced'
    ),
    'LogisticRegression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'roc_auc': roc_auc_score(y_test, y_prob),
        'f1': f1_score(y_test, y_pred),
    }
    print(f"  ROC-AUC: {results[name]['roc_auc']:.4f}, F1: {results[name]['f1']:.4f}")

## Evaluation

In [ ]:
# Detailed classification reports
for name, res in results.items():
    print(f"\n{'='*60}")
    print(f"{name}  (ROC-AUC: {res['roc_auc']:.4f})")
    print('='*60)
    print(classification_report(y_test, res['y_pred'], target_names=['No Booking', 'Booking']))

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Booking Prediction')
ax.legend()
plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
# Feature importance from the best tree-based model
best_name = max(
    [(n, r['roc_auc']) for n, r in results.items() if hasattr(r['model'], 'feature_importances_')],
    key=lambda x: x[1]
)[0]
best_model = results[best_name]['model']

importances = pd.Series(best_model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
top_features.plot(kind='barh', ax=ax)
ax.set_title(f'Top 15 Feature Importances ({best_name})')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## Inference

In [ ]:
# Run the best model on the inference sample
inference_df = pd.read_csv("../data/processed/inference_sample.csv")
inf_ids = inference_df[['user_id', 'timestamp', 'listing_id']].copy()
inf_X = inference_df.drop(columns=['user_id', 'timestamp', 'listing_id'])

# Align columns (inference sample may have different one-hot columns)
inf_X = inf_X.reindex(columns=X.columns, fill_value=0)
inf_X = inf_X.fillna(X.median())

# Predict booking probabilities
inf_ids['booking_probability'] = best_model.predict_proba(inf_X)[:, 1]

print(f"Inference predictions: {inf_ids.shape}")
print("\nProbability distribution:")
print(inf_ids['booking_probability'].describe())
inf_ids.sort_values('booking_probability', ascending=False).head(10)

In [ ]:
# Save the best model and inference predictions
joblib.dump(best_model, f"../data/processed/{best_name}_model.joblib")
joblib.dump(list(X.columns), "../data/processed/feature_columns.joblib")
inf_ids.to_csv("../data/processed/inference_predictions.csv", index=False)

print(f"Saved {best_name} model, feature columns, and predictions to data/processed/")